# Emerging Topics Guide

Surface what the market is *suddenly* talking about — topic clusters whose
article volume has spiked above their historical baseline.

**No API key required.** Reads from `data/topic_trends.tsv` (generated by CI).

## Sections
1. [Configuration](#1-configuration)
2. [Setup](#2-setup)
3. [Emerging topics for a date](#3-emerging-topics-for-a-date)
4. [Topic sentiment history](#4-topic-sentiment-history)
5. [All-dates spike calendar](#5-all-dates-spike-calendar)

---
## 1 — Configuration

In [ ]:
from datetime import date

DATE     = date.today().isoformat()   # "YYYY-MM-DD" — date to detect spikes for
TOP_N    = 10                         # max emerging topics to show
LOOKBACK = 45                         # days of history for trend chart

---
## 2 — Setup

In [ ]:
import sys
from datetime import date as _date, datetime
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..") if Path("../pipeline").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.cluster_topics import get_emerging_topics
from pipeline.constants import TOPIC_TRENDS_FILE

if not TOPIC_TRENDS_FILE.exists():
    raise FileNotFoundError(
        f"{TOPIC_TRENDS_FILE} not found.\n"
        "Run:  just cluster-topics   (or python enrich/cluster_topics.py)"
    )

trends_df = pd.read_csv(TOPIC_TRENDS_FILE, sep="\t")
print(f"topic_trends.tsv  : {len(trends_df):,} rows  |  {trends_df['date'].nunique()} dates")
print(f"Date range        : {trends_df['date'].min()} -> {trends_df['date'].max()}")
print(f"Unique topics     : {trends_df['topic_id'].nunique()}")

---
## 3 — Emerging topics for a date

`get_emerging_topics(date, trends_df)` compares each topic's article count on
`DATE` against its rolling baseline. Topics whose ratio exceeds the spike threshold
are returned, ranked by `spike_ratio` descending.

In [ ]:
run_date = datetime.strptime(DATE, "%Y-%m-%d").date()
spikes = get_emerging_topics(run_date, trends_df)

if not spikes:
    # Fall back to the most recent date that has spikes.
    for d in sorted(trends_df["date"].unique(), reverse=True):
        candidate = datetime.strptime(d, "%Y-%m-%d").date()
        spikes = get_emerging_topics(candidate, trends_df)
        if spikes:
            print(f"No spikes for {DATE} — showing {d} instead.")
            run_date = candidate
            break

if not spikes:
    print("No spike data found in topic_trends.tsv.")
else:
    print(f"Emerging topics for {run_date}  ({len(spikes)} spikes):\n")
    print(f"  {'#':<3} {'spike_ratio':>11}  {'articles':>8}  {'sentiment':>9}  label")
    print(f"  {'-'*75}")
    for i, s in enumerate(spikes[:TOP_N], 1):
        label = (s.get("label") or "(unlabeled)")[:45]
        sent  = s.get("sentiment_score", float("nan"))
        sent_str = f"{sent:>+.3f}" if sent == sent else "   n/a"
        print(f"  {i:<3} {s['spike_ratio']:>11.3f}  {s['article_count']:>8}  {sent_str}  {label}")

---
## 4 — Topic sentiment history

For any spike topic, plot its article count and sentiment score over the last
`LOOKBACK` days to see whether the spike is isolated or part of a sustained trend.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

if spikes:
    # Pick the top spike to chart; user can change index.
    top_spike = spikes[0]
    tid   = top_spike["topic_id"]
    label = (top_spike.get("label") or "(unlabeled)")[:55]

    topic_rows = trends_df[trends_df["topic_id"] == tid].copy()
    topic_rows["date"] = pd.to_datetime(topic_rows["date"])
    topic_rows = topic_rows.sort_values("date")

    # Restrict to LOOKBACK days.
    cutoff = topic_rows["date"].max() - pd.Timedelta(days=LOOKBACK)
    topic_rows = topic_rows[topic_rows["date"] >= cutoff]

    if topic_rows.empty:
        print(f"No history for topic '{label}' in the last {LOOKBACK} days.")
    else:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)

        # Article count.
        ax1.bar(topic_rows["date"], topic_rows["article_count"], color="#2c3e50", alpha=0.7)
        ax1.set_ylabel("Article count")
        ax1.set_title(f"{label}  |  spike={top_spike['spike_ratio']:.2f}x on {run_date}")

        # Sentiment score (may have NaN for pre-backfill rows).
        has_sentiment = topic_rows["sentiment_score"].notna()
        if has_sentiment.any():
            s_rows = topic_rows[has_sentiment]
            colors = ["#27ae60" if v > 0 else "#c0392b" if v < 0 else "#95a5a6"
                      for v in s_rows["sentiment_score"]]
            ax2.bar(s_rows["date"], s_rows["sentiment_score"], color=colors, alpha=0.7)
            ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
            ax2.set_ylabel("Sentiment score")
        else:
            ax2.text(0.5, 0.5, "No sentiment data (pre-backfill)", ha="center",
                     transform=ax2.transAxes, color="grey")

        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

---
## 5 — All-dates spike calendar

How many topics spiked on each date? Useful for spotting high-volatility periods.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date as _date, timedelta

# Compute spike count per date by replaying get_emerging_topics over all dates.
all_dates = sorted(trends_df["date"].unique())
spike_counts = []
for d in all_dates:
    dt = datetime.strptime(d, "%Y-%m-%d").date()
    n = len(get_emerging_topics(dt, trends_df))
    spike_counts.append({"date": pd.Timestamp(d), "n_spikes": n})

df_cal = pd.DataFrame(spike_counts)

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(df_cal["date"], df_cal["n_spikes"], color="#e74c3c", alpha=0.75)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45, ha="right")
ax.set_ylabel("Topics spiking")
ax.set_title("Topic spike calendar — all dates")
plt.tight_layout()
plt.show()

print(f"\nDates with 0 spikes : {(df_cal['n_spikes']==0).sum()}")
print(f"Mean spikes/day     : {df_cal['n_spikes'].mean():.1f}")
print(f"Max spikes in a day : {df_cal['n_spikes'].max()}  ({df_cal.loc[df_cal['n_spikes'].idxmax(), 'date'].date()})")